# 09. Opportunity-Gap Analysis (Study 2)

This notebook compares each hexagon's predicted café density against its actual density, using the final XGBoost model from Study 1. The goal is to identify where the model expects more popular cafés than currently exist (a potential opportunity gap), and where the reverse holds.

## Load Model and Full Dataset

Loading the final XGBoost (Poisson objective) model and the full modelling 
dataset, all 1,353 hexagons. Predictions are generated across the whole 
dataset, not just the test split, since this analysis uses the model to 
characterize expected café density citywide rather than test generalization 


In [1]:
import xgboost as xgb
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

gdf_full = gpd.read_file('../data/processed/berlin_h3_res8_with_all_features.geojson')

predictors = ['transit_stops_500m', 'rail_stations_800m', 'shops_500m', 
              'offices_500m', 'food_500m', 'universities_800m', 
              'coworking_500m', 'green_spaces_800m', 'culture_700m', 
              'population_density_per_km2']

print(f"Data shape: {gdf_full.shape}")
print('popular_cafe_count' in gdf_full.columns)

Data shape: (1353, 14)
True


The model in notebook 07 was trained on 929 hexagons and evaluated on 272 held-out hexagons which gives an honest measure of generalisation performance. The gap analysis needs different treatment, since every hexagon should be scored consistently - using the notebook 07 model directly would give train hexagons artificially small gaps and test hexagons artificially large gaps, purely because of which split they happened to land in.

To avoid this, I refit the same model configuration on all 1,353 hexagons, and all predictions andgaps below come from this full-data model. The generalisation performance reported earlier (Spearman 0.6022, RMSE 0.8972, MAE 0.3836) still comes from the notebook 07 test evaluation.

In [2]:
X_full = gdf_full[predictors]
y_full = gdf_full['popular_cafe_count']

xgb_full = xgb.XGBRegressor(
    objective='count:poisson',
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)
xgb_full.fit(X_full, y_full)

print("Model refit on full dataset (1,353 hexagons).")

Model refit on full dataset (1,353 hexagons).


##  Compute Predicted vs. Actual Gap

Generating predictions from the full-data model. The gap is predicted count minus actual count, where a positive gap suggests unrealised demand and a negative gap suggests saturation.

In [3]:
gdf_full['predicted_count'] = xgb_full.predict(X_full)
gdf_full['actual_count'] = gdf_full['popular_cafe_count']
gdf_full['gap'] = gdf_full['predicted_count'] - gdf_full['actual_count']

print(gdf_full[['predicted_count', 'actual_count', 'gap']].describe())

       predicted_count  actual_count          gap
count      1353.000000   1353.000000  1353.000000
mean          0.621204      0.623799    -0.002595
std           1.832242      2.012813     0.502245
min           0.006083      0.000000    -4.818808
25%           0.010133      0.000000     0.006340
50%           0.086500      0.000000     0.028869
75%           0.245560      0.000000     0.129687
max          18.884155     21.000000     3.830513


In [4]:
gdf_full.nsmallest(3, 'gap')[['h3_index', 'predicted_count', 'actual_count', 'gap']]

,h3_index,predicted_count,actual_count,gap
352,881f1d48a5fffff,10.181192,15,-4.818808
96,881f1d48d7fffff,6.970222,10,-3.029778
324,881f1d499bfffff,7.011680,10,-2.988320


## Standardize the Gap for Ranking

Raw gap values are the primary quantity of interest, but standardizing them gives a way to rank hexagons without needing an arbitrary cutoff.

In [5]:
gap_mean = gdf_full['gap'].mean()
gap_std = gdf_full['gap'].std()

gdf_full['gap_zscore'] = (gdf_full['gap'] - gap_mean) / gap_std

print(f"Gap mean: {gap_mean:.3f}, Gap std: {gap_std:.3f}")

Gap mean: -0.003, Gap std: 0.502


## Categorize Hexagons

Hexagons are labeled using standard deviation bands around the mean gap. 
This is a conventional threshold:

- Underserved: gap z-score > +1
- Matched: gap z-score between -1 and +1
- Oversaturated: gap z-score < -1

These categories exist for map visualization and discussion. The underlying continuous gap values remain the primary output.

In [6]:
def categorize_gap(z):
    if z > 1:
        return 'Underserved'
    elif z < -1:
        return 'Oversaturated'
    else:
        return 'Matched'

gdf_full['gap_category'] = gdf_full['gap_zscore'].apply(categorize_gap)

print(gdf_full['gap_category'].value_counts())

gap_category
Matched          1138
Oversaturated     142
Underserved        73
Name: count, dtype: int64


In [7]:
top_opportunities = gdf_full.nlargest(10, 'gap')[
    ['h3_index', 'predicted_count', 'actual_count', 'gap', 'gap_zscore']
]
print(top_opportunities)

             h3_index  predicted_count  actual_count       gap  gap_zscore
785   881f1d48cbfffff         4.830513             1  3.830513    7.631952
258   881f1d4ab3fffff         3.504128             1  2.504128    4.991039
183   881f1d49d5fffff         6.253766             4  2.253766    4.492553
1184  881f1d4d63fffff         2.229827             0  2.229827    4.444890
445   881f1d4d67fffff         6.152483             4  2.152483    4.290892
885   881f1d4d0dfffff         7.115298             5  2.115298    4.216856
164   881f1d49b1fffff         4.907776             3  1.907776    3.803666
228   881f1d4837fffff         6.630790             5  1.630790    3.252171
260   881f1d4f23fffff         1.623837             0  1.623837    3.238326
220   881f1d4ab1fffff         1.563280             0  1.563280    3.117753


The largest gaps cluster in hexagons with a strong predicted commercial and transit signal but low or zero actual café counts - matching RQ3, these read as candidate opportunity areas, where the model's spatial features suggest conditions favourable to café demand that the current landscape hasn't filled.

Whether a given case reflects a genuine opportunity or a factor outside the model's scope is a judgement call, though: zoning restrictions, high commercial rent, or recent closures are all possible explanations that the gap value alone can't settle. 

##  Save Output

Saving the full dataset with predictions, gap values, and categories for 
mapping and results reporting.

In [8]:
import joblib

joblib.dump(xgb_full, '../outputs/models/xgb_poisson_full_data.pkl')

OUT = '../data/processed/berlin_h3_opportunity_gap.geojson'
gdf_full.to_file(OUT, driver='GeoJSON')
print(f"Saved {len(gdf_full)} hexagons with gap analysis -> {OUT}")
print("Saved full-data model -> ../outputs/models/xgb_poisson_full_data.pkl")

Saved 1353 hexagons with gap analysis -> ../data/processed/berlin_h3_opportunity_gap.geojson
Saved full-data model -> ../outputs/models/xgb_poisson_full_data.pkl
